# Experiment No: 02  
## Implementation of Playfair Cipher

---

## Theory

The Playfair Cipher is a digraph substitution cipher. It encrypts pairs of letters instead of single letters. A 5×5 matrix is constructed using a keyword. The letters I and J are treated as the same character.

Rules:
1. If both letters are in the same row, replace each with the letter to its right.
2. If both letters are in the same column, replace each with the letter below it.
3. Otherwise, replace each letter with the letter in the same row but in the column of the other letter.

For repeated letters in a pair, a filler such as X is inserted.

---

## Algorithm

### Encryption
1. Generate the 5×5 key matrix using a keyword.
2. Convert plaintext to uppercase and remove spaces.
3. Replace J with I.
4. Divide plaintext into pairs.
5. Insert X between repeated letters in a pair.
6. If one letter remains, add X at the end.
7. Encrypt each pair using Playfair rules.

### Decryption
1. Take ciphertext in pairs.
2. Use reverse Playfair rules:
   - Same row → move left
   - Same column → move up
   - Rectangle → swap columns
3. Recover plaintext.

---

## Sample Output

```
Enter keyword: MONARCHY  
Enter plaintext: HELLO

Key Matrix:
M O N A R
C H Y B D
E F G I K
L P Q S T
U V W X Z
Encrypted text: CFSUPM
Decrypted text: HELXLO
```

In [ ]:
def generate_key_matrix(key):
    key = key.upper().replace("J", "I")
    seen = set()
    matrix_list = []

    # Add key letters
    for ch in key:
        if ch.isalpha() and ch not in seen:
            seen.add(ch)
            matrix_list.append(ch)

    # Add remaining letters
    for ch in "ABCDEFGHIKLMNOPQRSTUVWXYZ":
        if ch not in seen:
            matrix_list.append(ch)

    # Convert to 5x5 matrix
    matrix = [matrix_list[i:i+5] for i in range(0, 25, 5)]

    # Create position dictionary (for fast lookup)
    pos = {matrix[i][j]: (i, j) for i in range(5) for j in range(5)}

    return matrix, pos


def prepare_text(text):
    text = text.upper().replace("J", "I")
    text = "".join(ch for ch in text if ch.isalpha())

    result = ""
    i = 0

    while i < len(text):
        a = text[i]
        b = text[i + 1] if i + 1 < len(text) else 'X'

        if a == b:
            result += a + 'X'
            i += 1
        else:
            result += a + b
            i += 2

    if len(result) % 2 != 0:
        result += 'X'

    return result


def encrypt_playfair(text, matrix, pos):
    text = prepare_text(text)
    cipher = ""

    for i in range(0, len(text), 2):
        a, b = text[i], text[i + 1]
        r1, c1 = pos[a]
        r2, c2 = pos[b]

        if r1 == r2:  # Same row
            cipher += matrix[r1][(c1 + 1) % 5]
            cipher += matrix[r2][(c2 + 1) % 5]

        elif c1 == c2:  # Same column
            cipher += matrix[(r1 + 1) % 5][c1]
            cipher += matrix[(r2 + 1) % 5][c2]

        else:  # Rectangle
            cipher += matrix[r1][c2]
            cipher += matrix[r2][c1]

    return cipher


def decrypt_playfair(text, matrix, pos):
    text = "".join(ch for ch in text.upper() if ch.isalpha())
    plain = ""

    for i in range(0, len(text), 2):
        a, b = text[i], text[i + 1]
        r1, c1 = pos[a]
        r2, c2 = pos[b]

        if r1 == r2:  # Same row
            plain += matrix[r1][(c1 - 1) % 5]
            plain += matrix[r2][(c2 - 1) % 5]

        elif c1 == c2:  # Same column
            plain += matrix[(r1 - 1) % 5][c1]
            plain += matrix[(r2 - 1) % 5][c2]

        else:  # Rectangle
            plain += matrix[r1][c2]
            plain += matrix[r2][c1]

    return plain


# ================= MAIN =================
key = input("Enter keyword: ")
plaintext = input("Enter plaintext: ")

matrix, pos = generate_key_matrix(key)

print("\nKey Matrix:")
for row in matrix:
    print(" ".join(row))

cipher = encrypt_playfair(plaintext, matrix, pos)
print("\nEncrypted text:", cipher)

decrypted = decrypt_playfair(cipher, matrix, pos)
print("Decrypted text:", decrypted)

Key Matrix:
M O N A R
C H Y B D
E F G I K
L P Q S T
U V W X Z
Encrypted text: CFSUPM
Decrypted text: HELXLO
